In [2]:
!pip install requests

In [3]:
import requests
import os
import json
import time
from datetime import datetime

In [4]:
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

HEADERS = {
    "User-Agent": "TrendPulse/1.0"
}

CATEGORY_KEYWORDS = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

In [5]:
import requests
import os
import json
import time
from datetime import datetime

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

HEADERS = {
    "User-Agent": "TrendPulse/1.0"
}

# Slightly expanded keyword variants
CATEGORY_KEYWORDS = {
    "technology": [
        "ai", "software", "tech", "code", "computer", "data",
        "cloud", "api", "gpu", "llm", "model", "ml"
    ],
    "worldnews": [
        "war", "government", "country", "president", "election",
        "elections", "climate", "attack", "global", "minister", "policy"
    ],
    "sports": [
        "nfl", "nba", "fifa", "sport", "sports", "game", "games",
        "team", "player", "players", "league", "championship", "match"
    ],
    "science": [
        "research", "study", "studies", "space", "physics", "biology",
        "discovery", "discoveries", "nasa", "genome", "scientist", "science"
    ],
    "entertainment": [
        "movie", "movies", "film", "music", "netflix", "game", "games",
        "book", "books", "show", "shows", "award", "awards", "streaming"
    ]
}

def get_top_story_ids():
    try:
        response = requests.get(TOP_STORIES_URL, headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()[:500]
    except requests.RequestException as e:
        print("Error fetching top story IDs:", e)
        return []

def get_story_details(story_id):
    try:
        response = requests.get(ITEM_URL.format(story_id), headers=HEADERS, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"Error fetching story {story_id}: {e}")
        return None

def title_matches_category(title, category):
    if not title:
        return False

    title = title.lower()

    for keyword in CATEGORY_KEYWORDS[category]:
        if keyword in title:
            return True

    return False

def extract_story_fields(story, category):
    return {
        "post_id": story.get("id"),
        "title": story.get("title", ""),
        "category": category,
        "score": story.get("score", 0),
        "num_comments": story.get("descendants", 0),
        "author": story.get("by", "unknown"),
        "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

def save_to_json(data):
    os.makedirs("data", exist_ok=True)
    filename = f"trends_{datetime.now().strftime('%Y%m%d')}.json"
    filepath = os.path.join("data", filename)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    return filepath

def main():
    story_ids = get_top_story_ids()

    if not story_ids:
        print("No story IDs found.")
        return

    collected_stories = []

    category_counts = {
        "technology": 0,
        "worldnews": 0,
        "sports": 0,
        "science": 0,
        "entertainment": 0
    }

    category_order = ["technology", "worldnews", "sports", "science", "entertainment"]

    # Process each category independently
    for category in category_order:
        print(f"\nProcessing: {category}")

        for story_id in story_ids:
            if category_counts[category] >= 25:
                break

            story = get_story_details(story_id)

            if not story or story.get("type") != "story":
                continue

            title = story.get("title", "")

            if title_matches_category(title, category):
                story_data = extract_story_fields(story, category)
                collected_stories.append(story_data)
                category_counts[category] += 1

        print(f"Collected in {category}: {category_counts[category]}")

        if category != category_order[-1]:
            time.sleep(2)

    filepath = save_to_json(collected_stories)

    print("\nCategory Counts:", category_counts)
    print(f"Collected {len(collected_stories)} stories")
    print("Saved to:", filepath)

main()


Processing: technology
Collected in technology: 25

Processing: worldnews
Collected in worldnews: 25

Processing: sports
Collected in sports: 15

Processing: science
Collected in science: 20

Processing: entertainment
Collected in entertainment: 25

Category Counts: {'technology': 25, 'worldnews': 25, 'sports': 15, 'science': 20, 'entertainment': 25}
Collected 110 stories
Saved to: data/trends_20260406.json


In [8]:
import json

with open("data/trends_20260406.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total stories:", len(data))
print(data[0])

Total stories: 110
{'post_id': 47655408, 'title': 'Show HN: I built a tiny LLM to demystify how language models work', 'category': 'technology', 'score': 625, 'num_comments': 80, 'author': 'armanified', 'collected_at': '2026-04-06 13:16:06'}


In [9]:
required_fields = ["post_id", "title", "category", "score", "num_comments", "author", "collected_at"]

for i, story in enumerate(data):
    for field in required_fields:
        if field not in story:
            print(f"Missing field '{field}' in story index {i}")

In [10]:
category_counts = {}

for story in data:
    category = story["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

print(category_counts)

{'technology': 25, 'worldnews': 25, 'sports': 15, 'science': 20, 'entertainment': 25}
